Training and validation were performed with the SwissProt Training/Validation dataset used to train the DeepLoc 2.0 server. Testing was performed using the DeepLoc 2.0 Human Protein Atlas (HPA) test set. The data is available at: https://services.healthtech.dtu.dk/services/DeepLoc-2.0/4-Data.php

**References**

Thumuluri, V., Almagro Armenteros, J. J., Johansen, A. R., Nielsen, H. & Winther, O. (2022). *DeepLoc 2.0: multi-label subcellular localization prediction using protein language models.* Nucleic Acids Research, 50(W1), W228–W234. https://doi.org/10.1093/nar/gkac278

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

In [2]:
#exploring DeepLoc 2.0 Train/Validation set for multi-label subcellular localisation prediction 
df = pd.read_csv("Swissprot_Train_Validation_dataset.csv")
df.head()

,Unnamed: 0,ACC,Kingdom,Partition,Membrane,Cytoplasm,Nucleus,Extracellular,Cell membrane,Mitochondrion,Plastid,Endoplasmic reticulum,Lysosome/Vacuole,Golgi apparatus,Peroxisome,Sequence
0,0,Q28165,Metazoa,4,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...
1,1,Q86U42,Metazoa,4,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...
2,2,Q0GA42,Metazoa,3,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAALGVRLRDCCSRGAVLLLFFSLSPRPPAAAAWLLGLR...
3,3,P82349,Metazoa,1,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAATEQQGSNGPVKKSMREKAVERRNVNKEHNSNFKAGY...
4,4,Q7L5N1,Metazoa,1,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MAAAAAAAAATNGTGGSSGMEVDAAVVPSVMACGVTGSVSVALHPL...


In [3]:
df.shape #dimensions

(28303, 16)

In [4]:
df = df.dropna(subset = ["Sequence", "Nucleus"])
df.shape

(28303, 16)

In [5]:
df.index #no. of training examples

RangeIndex(start=0, stop=28303, step=1)

In [6]:
df.columns

Index(['Unnamed: 0', 'ACC', 'Kingdom', 'Partition', 'Membrane', 'Cytoplasm',
       'Nucleus', 'Extracellular', 'Cell membrane', 'Mitochondrion', 'Plastid',
       'Endoplasmic reticulum', 'Lysosome/Vacuole', 'Golgi apparatus',
       'Peroxisome', 'Sequence'],
      dtype='object')

In [7]:
df["Nucleus"].unique() #confirm use of 0/1 labels

array([1., 0.])

In [8]:
df["Nucleus"].value_counts()

Nucleus
0.0    18583
1.0     9720
Name: count, dtype: int64

In [9]:
df["Sequence"].str.len().describe() #stats for lengths of proteins

count    28303.000000
mean       556.901141
std        558.491431
min         40.000000
25%        261.000000
50%        433.000000
75%        681.000000
max      34350.000000
Name: Sequence, dtype: float64

In [10]:
df["Kingdom"].unique() #all eukaryotic according to https://doi.org/10.1093/nar/gkac278 (DeepLoc 2.0 paper)

array(['Metazoa', 'Viridiplantae', 'Fungi', 'Other'], dtype=object)

In [11]:
df["Kingdom"].value_counts()

Kingdom
Metazoa          16741
Fungi             5841
Viridiplantae     5116
Other              605
Name: count, dtype: int64

In [12]:
df["Partition"].unique()

array([4, 3, 1, 0, 2])

In [13]:
df["Partition"].value_counts()

Partition
0    5963
2    5731
3    5696
4    5462
1    5451
Name: count, dtype: int64

In [14]:
#produce the relevant training/validation dataframe
df = df[["Kingdom", "Partition", "Nucleus", "Sequence"]]
df.head()

,Kingdom,Partition,Nucleus,Sequence
0,Metazoa,4,1.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...
1,Metazoa,4,1.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...
2,Metazoa,3,0.0,MAAAAAAAAALGVRLRDCCSRGAVLLLFFSLSPRPPAAAAWLLGLR...
3,Metazoa,1,0.0,MAAAAAAAAATEQQGSNGPVKKSMREKAVERRNVNKEHNSNFKAGY...
4,Metazoa,1,1.0,MAAAAAAAAATNGTGGSSGMEVDAAVVPSVMACGVTGSVSVALHPL...


In [15]:
df.shape

(28303, 4)

In [16]:
#exploring DeepLoc 2.0 independent test set for multi-label subcellular localisation prediction 
df_test = pd.read_csv("hpa_testset.csv")
df_test.head()

,sid,Cell membrane,Cytoplasm,Endoplasmic reticulum,Golgi apparatus,Lysosome/Vacuole,Mitochondrion,Nucleus,Peroxisome,Lengths,fasta
0,ENSP00000355743,0,0,0,0,0,0,1,0,496,MAAAAGPGAALSPRPCDSDPATPGAQSPKDDNEDNSNDGTQPSKRR...
1,ENSP00000429628,0,0,0,0,0,0,1,0,173,MAAAALRDPAQVPVAADLLTDHEEGYVTFEDVAVYFSQEEWRLLDD...
2,ENSP00000266544,0,0,0,0,0,1,0,0,377,MAAAAQSRVVRVLSMSRSAITAIATSVCHGPPCRQLHHALMPHGKG...
3,ENSP00000470652,0,0,0,1,0,0,0,0,410,MAAAAVGAGHGAGGPGAASSSGGAREGARVAALCLLWYALSAGGNV...
4,ENSP00000460751,0,0,0,0,0,0,1,0,163,MAAAAVTRGTPGENSHHLKIFLPKKLLECLPRCPLLPPERLRWNTN...


In [17]:
df_test.shape

(1717, 11)

In [18]:
df_test.index

RangeIndex(start=0, stop=1717, step=1)

In [19]:
df_test.columns

Index(['sid', 'Cell membrane', 'Cytoplasm', 'Endoplasmic reticulum',
       'Golgi apparatus', 'Lysosome/Vacuole', 'Mitochondrion', 'Nucleus',
       'Peroxisome', 'Lengths', 'fasta'],
      dtype='object')

In [20]:
df_test.rename(columns={"fasta": "Sequence"}, inplace=True)
df_test = df_test.dropna(subset = ["Sequence", "Nucleus"])
df_test.shape

(1717, 11)

In [21]:
df_test.columns

Index(['sid', 'Cell membrane', 'Cytoplasm', 'Endoplasmic reticulum',
       'Golgi apparatus', 'Lysosome/Vacuole', 'Mitochondrion', 'Nucleus',
       'Peroxisome', 'Lengths', 'Sequence'],
      dtype='object')

In [22]:
df_test["Nucleus"].unique()

array([1, 0])

In [23]:
df_test["Nucleus"].value_counts()

Nucleus
1    893
0    824
Name: count, dtype: int64

In [24]:
df_test["Lengths"].describe() #shows test sequences are shorter on average & max length is also much shorter

count    1717.000000
mean      294.166570
std       373.415303
min        23.000000
25%        98.000000
50%       170.000000
75%       339.000000
max      4834.000000
Name: Lengths, dtype: float64

In [25]:
#produce relevant test set dataframe
df_test = df_test[["Nucleus", "Sequence"]]
df_test.head()

,Nucleus,Sequence
0,1,MAAAAGPGAALSPRPCDSDPATPGAQSPKDDNEDNSNDGTQPSKRR...
1,1,MAAAALRDPAQVPVAADLLTDHEEGYVTFEDVAVYFSQEEWRLLDD...
2,0,MAAAAQSRVVRVLSMSRSAITAIATSVCHGPPCRQLHHALMPHGKG...
3,0,MAAAAVGAGHGAGGPGAASSSGGAREGARVAALCLLWYALSAGGNV...
4,1,MAAAAVTRGTPGENSHHLKIFLPKKLLECLPRCPLLPPERLRWNTN...


In [26]:
#feature engineering - amino acid composition (as a fraction of sequence length)
#note: this method of feature engineering loses information about amino acid positions
aminoacids = {"A":0, "R":1, "N":2, "D":3, "C":4, "Q":5, "E":6, "G":7, "H":8, "I":9, "L":10, "K":11, "M":12, "F":13, "P":14, "S":15, "T":16, "W":17, "Y":18, "V":19}
def aminoacid_composition(sequence):
    features = np.zeros(20)
    length = len(sequence)
    for aa in sequence:
        for symbol in aminoacids:
            if aa == symbol:
                features[aminoacids[symbol]] += (1 / length)
    return features

df["input_vectors"] = df["Sequence"].apply(aminoacid_composition)

In [27]:
df.head()

,Kingdom,Partition,Nucleus,Sequence,input_vectors
0,Metazoa,4,1.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...,"[0.11437908496732031, 0.09150326797385624, 0.0..."
1,Metazoa,4,1.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...,"[0.11437908496732031, 0.09150326797385624, 0.0..."
2,Metazoa,3,0.0,MAAAAAAAAALGVRLRDCCSRGAVLLLFFSLSPRPPAAAAWLLGLR...,"[0.08937960042060973, 0.07781282860147205, 0.0..."
3,Metazoa,1,0.0,MAAAAAAAAATEQQGSNGPVKKSMREKAVERRNVNKEHNSNFKAGY...,"[0.05937500000000002, 0.05000000000000001, 0.0..."
4,Metazoa,1,1.0,MAAAAAAAAATNGTGGSSGMEVDAAVVPSVMACGVTGSVSVALHPL...,"[0.08562691131498473, 0.03669724770642202, 0.0..."


In [28]:
df_test["input_vectors"] = df_test["Sequence"].apply(aminoacid_composition)
df_test.head()

,Nucleus,Sequence,input_vectors
0,1,MAAAAGPGAALSPRPCDSDPATPGAQSPKDDNEDNSNDGTQPSKRR...,"[0.05443548387096771, 0.060483870967741896, 0...."
1,1,MAAAALRDPAQVPVAADLLTDHEEGYVTFEDVAVYFSQEEWRLLDD...,"[0.09826589595375719, 0.06358381502890172, 0.0..."
2,0,MAAAAQSRVVRVLSMSRSAITAIATSVCHGPPCRQLHHALMPHGKG...,"[0.07957559681697617, 0.07692307692307696, 0.0..."
3,0,MAAAAVGAGHGAGGPGAASSSGGAREGARVAALCLLWYALSAGGNV...,"[0.09512195121951224, 0.04390243902439025, 0.0..."
4,1,MAAAAVTRGTPGENSHHLKIFLPKKLLECLPRCPLLPPERLRWNTN...,"[0.06748466257668712, 0.07975460122699388, 0.0..."


In [29]:
#to use for separate "fromscratch_logistic_regression" exercise
df.to_csv("Protein_localisation_data/Training_CV_set.csv")
df_test.to_csv("Protein_localisation_data/Test_set.csv")

In [30]:
#logistic regression model with 5-fold cross-validation across homology-partitioned training datasets
#first fold of 5-fold cross-validation will use homology partition 0 as the validation set
train_df = df[df["Partition"].isin([1, 2, 3, 4])]
X_train = np.stack(train_df["input_vectors"].values)
y_train = np.stack(train_df["Nucleus"].values)

cv_df = df[df["Partition"] == 0]
X_cv = np.stack(cv_df["input_vectors"].values)
y_cv = np.stack(cv_df["Nucleus"].values)

X_test = np.stack(df_test["input_vectors"].values)
y_test = np.stack(df_test["Nucleus"].values)

In [31]:
#feature scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_cv_scaled = scaler.transform(X_cv)
X_test_scaled = scaler.transform(X_test)

In [32]:
#initialise the class
model = LogisticRegression()
#train the model
model.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

In [33]:
yhat = model.predict(X_cv_scaled)

#extract probability of output label 1
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
print(y_prob[0:200])

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

[0.63504313 0.26246396 0.22454369 0.09369078 0.16848342 0.21722055
 0.20912252 0.11049553 0.38125878 0.53314242 0.59546943 0.45895212
 0.14684296 0.13145989 0.18182744 0.38944828 0.4284688  0.14756092
 0.09878188 0.10706057 0.16730647 0.350556   0.54751671 0.51407833
 0.1731836  0.17552573 0.39587241 0.64404818 0.60058741 0.27471217
 0.11688224 0.14253809 0.04164947 0.23072247 0.23298356 0.23431967
 0.13986051 0.31703198 0.43331889 0.28708683 0.31390959 0.39083081
 0.30408598 0.09174857 0.42900763 0.16351228 0.12987442 0.38754149
 0.41999415 0.32288697 0.25562362 0.35403781 0.14873381 0.16273142
 0.69450251 0.68412564 0.0955861  0.14631229 0.17579966 0.23305231
 0.27742987 0.1423244  0.39059602 0.32251035 0.28614047 0.20953763
 0.21211737 0.43793618 0.44420996 0.18340265 0.72684999 0.49054425
 0.63497073 0.74961595 0.50899534 0.48162848 0.31088942 0.38808097
 0.69263213 0.69180339 0.70421833 0.47502569 0.45329709 0.85215049
 0.85336553 0.18966673 0.15716123 0.14982186 0.17626332 0.2931

In [34]:
np.mean(y_cv)

np.float64(0.31376823746436355)

Accuracy is not a useful metric for the model's performance since the data is skewed (35% nuclear proteins in the cross-validation set, so approximately 27% being misclassified is only a marginally better outcome than if every protein was predicted to be non-nuclear).

For class 0, precision = 0.76, and for class 1, precision = 0.60, representing that the model correctly identifies a reasonably good fraction of cases that are actually 0/1. For class 0, recall = 0.88, whereas for class 1, recall = 0.40. The model is therefore significantly more accurate in identifying examples where y = 0 than y = 1, resulting in a high proportion of false negatives. This could be improved by lowering the threshold below 0.5, although this will involve a trade-off between precision and recall: precision will decrease, while recall increases. 

In [35]:
#lower the threshold to 0.4
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
threshold = 0.40
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.7174241153781653
              precision    recall  f1-score   support

         0.0       0.80      0.78      0.79      4092
         1.0       0.55      0.58      0.56      1871

    accuracy                           0.72      5963
   macro avg       0.67      0.68      0.68      5963
weighted avg       0.72      0.72      0.72      5963

[[3192  900]
 [ 785 1086]]


In [36]:
#lower the threshold to 0.3
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
threshold = 0.3
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.6649337581754151
              precision    recall  f1-score   support

         0.0       0.85      0.62      0.72      4092
         1.0       0.48      0.76      0.59      1871

    accuracy                           0.66      5963
   macro avg       0.66      0.69      0.65      5963
weighted avg       0.73      0.66      0.68      5963

[[2542 1550]
 [ 448 1423]]


In [37]:
#2nd fold of 5-fold cross-validation will use homology partition 1 as the validation set
train_df = df[df["Partition"].isin([0, 2, 3, 4])]
X_train = np.stack(train_df["input_vectors"].values)
y_train = np.stack(train_df["Nucleus"].values)

cv_df = df[df["Partition"] == 1]
X_cv = np.stack(cv_df["input_vectors"].values)
y_cv = np.stack(cv_df["Nucleus"].values)

In [38]:
X_train_scaled = scaler.fit_transform(X_train)
X_cv_scaled = scaler.transform(X_cv)

#initialise the class
model = LogisticRegression()
#train the model
model.fit(X_train_scaled, y_train)

yhat = model.predict(X_cv_scaled)

#extract probability of output label 1
y_prob = model.predict_proba(X_cv_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.6917996697853604
              precision    recall  f1-score   support

         0.0       0.72      0.85      0.78      3536
         1.0       0.59      0.40      0.48      1915

    accuracy                           0.69      5451
   macro avg       0.66      0.62      0.63      5451
weighted avg       0.68      0.69      0.67      5451

[[3010  526]
 [1154  761]]


In [39]:
#lower the threshold to 0.4
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
threshold = 0.40
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.6936341955604476
              precision    recall  f1-score   support

         0.0       0.77      0.76      0.76      3536
         1.0       0.56      0.57      0.57      1915

    accuracy                           0.69      5451
   macro avg       0.66      0.67      0.66      5451
weighted avg       0.69      0.69      0.69      5451

[[2690  846]
 [ 824 1091]]


In [40]:
#lower the threshold to 0.3
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
threshold = 0.3
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.6525408181984957
              precision    recall  f1-score   support

         0.0       0.81      0.60      0.69      3536
         1.0       0.50      0.74      0.60      1915

    accuracy                           0.65      5451
   macro avg       0.66      0.67      0.65      5451
weighted avg       0.70      0.65      0.66      5451

[[2136 1400]
 [ 494 1421]]


In [41]:
#3rd fold of 5-fold cross-validation will use homology partition 2 as the validation set
train_df = df[df["Partition"].isin([0, 1, 3, 4])]
X_train = np.stack(train_df["input_vectors"].values)
y_train = np.stack(train_df["Nucleus"].values)

cv_df = df[df["Partition"] == 2]
X_cv = np.stack(cv_df["input_vectors"].values)
y_cv = np.stack(cv_df["Nucleus"].values)

X_train_scaled = scaler.fit_transform(X_train)
X_cv_scaled = scaler.transform(X_cv)

#initialise the class
model = LogisticRegression()
#train the model
model.fit(X_train_scaled, y_train)

yhat = model.predict(X_cv_scaled)

#extract probability of output label 1
y_prob = model.predict_proba(X_cv_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.7253533414761821
              precision    recall  f1-score   support

         0.0       0.75      0.87      0.81      3800
         1.0       0.64      0.43      0.51      1931

    accuracy                           0.73      5731
   macro avg       0.69      0.65      0.66      5731
weighted avg       0.71      0.73      0.71      5731

[[3323  477]
 [1097  834]]


In [42]:
#lower the threshold to 0.4
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
threshold = 0.40
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.722212528354563
              precision    recall  f1-score   support

         0.0       0.80      0.78      0.79      3800
         1.0       0.58      0.61      0.60      1931

    accuracy                           0.72      5731
   macro avg       0.69      0.69      0.69      5731
weighted avg       0.73      0.72      0.72      5731

[[2960  840]
 [ 752 1179]]


In [43]:
#lower the threshold to 0.3
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
threshold = 0.3
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.6756238003838771
              precision    recall  f1-score   support

         0.0       0.85      0.62      0.72      3800
         1.0       0.51      0.79      0.62      1931

    accuracy                           0.68      5731
   macro avg       0.68      0.70      0.67      5731
weighted avg       0.74      0.68      0.68      5731

[[2353 1447]
 [ 412 1519]]


In [44]:
#4th fold of 5-fold cross-validation will use homology partition 3 as the validation set
train_df = df[df["Partition"].isin([0, 1, 2, 4])]
X_train = np.stack(train_df["input_vectors"].values)
y_train = np.stack(train_df["Nucleus"].values)

cv_df = df[df["Partition"] == 3]
X_cv = np.stack(cv_df["input_vectors"].values)
y_cv = np.stack(cv_df["Nucleus"].values)

X_train_scaled = scaler.fit_transform(X_train)
X_cv_scaled = scaler.transform(X_cv)

#initialise the class
model = LogisticRegression()
#train the model
model.fit(X_train_scaled, y_train)

yhat = model.predict(X_cv_scaled)

#extract probability of output label 1
y_prob = model.predict_proba(X_cv_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.7008426966292135
              precision    recall  f1-score   support

         0.0       0.73      0.87      0.79      3710
         1.0       0.61      0.39      0.47      1986

    accuracy                           0.70      5696
   macro avg       0.67      0.63      0.63      5696
weighted avg       0.69      0.70      0.68      5696

[[3226  484]
 [1220  766]]


In [45]:
#lower the threshold to 0.4
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
threshold = 0.40
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.6966292134831461
              precision    recall  f1-score   support

         0.0       0.77      0.77      0.77      3710
         1.0       0.57      0.56      0.56      1986

    accuracy                           0.70      5696
   macro avg       0.67      0.66      0.67      5696
weighted avg       0.70      0.70      0.70      5696

[[2857  853]
 [ 875 1111]]


In [46]:
#lower the threshold to 0.3
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
threshold = 0.3
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.6641502808988764
              precision    recall  f1-score   support

         0.0       0.83      0.61      0.70      3710
         1.0       0.51      0.76      0.61      1986

    accuracy                           0.66      5696
   macro avg       0.67      0.69      0.66      5696
weighted avg       0.72      0.66      0.67      5696

[[2264 1446]
 [ 467 1519]]


In [47]:
#5th fold of 5-fold cross-validation will use homology partition 4 as the validation set
train_df = df[df["Partition"].isin([0, 1, 2, 3])]
X_train = np.stack(train_df["input_vectors"].values)
y_train = np.stack(train_df["Nucleus"].values)

cv_df = df[df["Partition"] == 4]
X_cv = np.stack(cv_df["input_vectors"].values)
y_cv = np.stack(cv_df["Nucleus"].values)

X_train_scaled = scaler.fit_transform(X_train)
X_cv_scaled = scaler.transform(X_cv)

#initialise the class
model = LogisticRegression()
#train the model
model.fit(X_train_scaled, y_train)

yhat = model.predict(X_cv_scaled)

#extract probability of output label 1
y_prob = model.predict_proba(X_cv_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.6999267667521054
              precision    recall  f1-score   support

         0.0       0.71      0.88      0.79      3445
         1.0       0.66      0.39      0.49      2017

    accuracy                           0.70      5462
   macro avg       0.68      0.64      0.64      5462
weighted avg       0.69      0.70      0.68      5462

[[3027  418]
 [1221  796]]


In [48]:
#lower the threshold to 0.4
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
threshold = 0.40
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.7010252654705236
              precision    recall  f1-score   support

         0.0       0.75      0.78      0.77      3445
         1.0       0.60      0.57      0.58      2017

    accuracy                           0.70      5462
   macro avg       0.68      0.67      0.68      5462
weighted avg       0.70      0.70      0.70      5462

[[2686  759]
 [ 874 1143]]


In [49]:
#lower the threshold to 0.3
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
threshold = 0.3
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.6724642987916514
              precision    recall  f1-score   support

         0.0       0.82      0.62      0.70      3445
         1.0       0.54      0.77      0.63      2017

    accuracy                           0.67      5462
   macro avg       0.68      0.69      0.67      5462
weighted avg       0.72      0.67      0.68      5462

[[2128 1317]
 [ 472 1545]]


Classification reports and confusion matrices are very similar across 5-fold cross-validation, suggesting that the initial plan to improve recall through a lower threshold would be effective across all folds. Lowering the threshold to 0.4 successfully improved the f1 score across folds from ~0.49 to ~0.57. An even lower threshold was avoided in the final model, as a threshold of 0.3 reduced precision to ~0.51.

In [50]:
#C value = inverse of regularisation strength in scikit-learn
C_values = [0.01, 0.1, 1, 10, 100]
accuracies = []
f1_scores = []

#cross-validation for different C-values using fold 5
for c in C_values:
    model = LogisticRegression(C=c)
    model.fit(X_train_scaled, y_train)
    y_prob = model.predict_proba(X_cv_scaled)[:, 1]
    threshold = 0.40
    yhat = (y_prob >= threshold).astype(int)
    accuracy = accuracy_score(y_cv, yhat)
    accuracies.append(accuracy)
    print(f"C={c}, Accuracy: {accuracy:.3f}")
    f1 = f1_score(y_cv, yhat)
    f1_scores.append(f1)
    print(f"C={c}, f1: {f1:.3f}")

C=0.01, Accuracy: 0.701
C=0.01, f1: 0.583
C=0.1, Accuracy: 0.701
C=0.1, f1: 0.583
C=1, Accuracy: 0.701
C=1, f1: 0.583
C=10, Accuracy: 0.701
C=10, f1: 0.583
C=100, Accuracy: 0.701
C=100, f1: 0.583


In [51]:
#final model
X_train = np.stack(df["input_vectors"].values)
y_train = np.stack(df["Nucleus"].values)

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(C = 1.0)
model.fit(X_train_scaled, y_train)

y_prob = model.predict_proba(X_test_scaled)[:, 1]
threshold = 0.4
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_test, yhat))
print(classification_report(y_test, yhat))
print(confusion_matrix(y_test, yhat))

Accuracy: 0.5899825276645312
              precision    recall  f1-score   support

           0       0.56      0.66      0.61       824
           1       0.63      0.53      0.57       893

    accuracy                           0.59      1717
   macro avg       0.59      0.59      0.59      1717
weighted avg       0.60      0.59      0.59      1717

[[542 282]
 [422 471]]


In [52]:
y_prob = model.predict_proba(X_train_scaled)[:, 1]
threshold = 0.4
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_train, yhat))
print(classification_report(y_train, yhat))
print(confusion_matrix(y_train, yhat))

Accuracy: 0.7095360915803979
              precision    recall  f1-score   support

         0.0       0.78      0.78      0.78     18583
         1.0       0.58      0.58      0.58      9720

    accuracy                           0.71     28303
   macro avg       0.68      0.68      0.68     28303
weighted avg       0.71      0.71      0.71     28303

[[14442  4141]
 [ 4080  5640]]
